<style>
    body {
        font-family: 'Arial', sans-serif;
        background-color: #f4f4f4;
        color: #333;
        line-height: 1.6;
    }

    .main-container {
        max-width: 940px;
        margin: 20px auto;
        background-color: #ffffff;
        border-radius: 8px;
        box-shadow: 0 2px 10px rgba(0, 0, 0, 0.1);
        padding: 20px;
    }

    h1.title {
        font-size: 3em;
        color: #0056b3; /* Darker blue for a more professional look */
        margin-bottom: 20px;
        font-weight: bold; /* Bold for emphasis */
        text-align: center; /* Center the title */
    }

    h2 {
        font-size: 2em;
        color: #007bff; /* Lighter blue for subheadings */
        margin-top: 40px;
        margin-bottom: 10px;
        font-weight: 600; /* Semi-bold for better visibility */
        border-bottom: 2px solid #e9ecef; /* Underline for emphasis */
        padding-bottom: 5px; /* Space below the heading */
    }

    h3 {
        font-size: 1.75em;
        color: #0056b3;
        margin-top: 20px;
        margin-bottom: 10px;
        font-weight: 500; /* Medium weight for sub-subheadings */
    }

    p {
        font-size: 1.2em;
        margin-bottom: 10px;
        line-height: 1.5; /* Increased line height for better readability */
    }

    pre {
        background-color: #f8f9fa;
        padding: 15px;
        border-radius: 5px;
        overflow-x: auto;
        border: 1px solid #ddd;
    }

    code {
        background-color: #e9ecef;
        padding: 2px 4px;
        border-radius: 4px;
    }

    .dataframe-container {
        overflow-x: auto; /* Enable horizontal scrolling */
        margin-bottom: 20px; /* Space below the DataFrame */
        border: 1px solid #ddd; /* Optional border for better visibility */
        border-radius: 5px; /* Rounded corners */
    }

    table {
        width: 100%; /* Ensure the table takes full width */
        border-collapse: collapse; /* Remove space between cells */
    }

    th, td {
        padding: 10px; /* Padding for table cells */
        text-align: left; /* Align text to the left */
        border-bottom: 1px solid #ddd; /* Light border for separation */
    }

    th {
        background-color: #ffffff; /* Change header background color to white */
        color: #333; /* Header text color */
        font-weight: bold; /* Bold header text */
    }

    tr:hover {
        background-color: #f1f1f1; /* Highlight row on hover */
    }

    .footer {
        text-align: center;
        margin-top: 20px;
        font-size: 0.9em;
        color: #6c757d;
    }

    /* Responsive adjustments */
    @media (max-width: 768px) {
        .main-container {
            padding: 15px;
        }

        h1.title {
            font-size: 2em;
        }

        h2, h3 {
            font-size: 1.5em;
        }
    }
</style>

## Introduction

In this notebook similarity search is used to pick the best 25 possible misconception for a given question. 

Its done by concatenating values of all text columns in the `test` dataset and then embedding these concatenated strings.

I also embedd all misconceptions from the `misconception_mapping` dataset.

Then by performing semantic search on these embedding, I select the closest 25 misconception for a given concatenated string.

The model used to embedd these text values is bge-large-en-1.5 model which has been fine-tuned beforehand (fine-tuning of the model for embedding explained in `fine_tuning_bge_large.html`).


In [ ]:
import pandas as pd
import numpy as np
import torch
import os
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

# Display the DataFrame in a scrollable container
from IPython.core.display import display, HTML

## Load data
Loading test dataset from the train-test split of training data (this split explained at the end of `data_preprocessing_to_render.html`).

In [4]:
#IS_SUBMISSION = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))
df = pd.read_csv("./data/train_df_test.csv")

misconception_mapping = pd.read_csv("./data/misconception_mapping.csv")

### Test Dataset Example

In [5]:
display(HTML(f'<div class="dataframe-container">{df.head(3).to_html(index=False)}</div>'))

QuestionId,WholeAssignment,ConstructName,SubjectName,QuestionText,CorrectAnswer,ConstructId,SubjectId,Answer,Answer_value,Misconception,MisconceptionId
1758,Express a non-linear equation as a function machine. Function Machines. Which function machine matches the equation \( y=x^{2}+4 ? \),Express a non-linear equation as a function machine,Function Machines,Which function machine matches the equation \( y=x^{2}+4 ? \),C,1508,649,AnswerDText,"![A function machine which has 4 parts joined by arrows pointing from left to right. ""y"" is the first part, written on the left, followed by a horizontal arrow pointing to a rectangle that has ""+ 4"" written inside it, followed by a horizontal arrow pointing to a rectangle that has ""square"" written inside it, followed by a horizontal arrow pointing to ""𝑥""]()",MisconceptionDId,2389.0
1426,Identify a unit of length. Volume and Capacity Units. George has answered a question on perimeter and worked out an answer of \( 310 \).\n\nBehind the star he has written the units that he used.\n\nWhich of the following units is definitely wrong? \( 310 \color{yellow} \bigstar\),Identify a unit of length,Volume and Capacity Units,George has answered a question on perimeter and worked out an answer of \( 310 \).\n\nBehind the star he has written the units that he used.\n\nWhich of the following units is definitely wrong? \( 310 \color{yellow} \bigstar\),D,2012,197,AnswerCText,\( f t \),MisconceptionCId,719.0
1181,Identify the net of a prism. Nets. Which is the correct net for this shape? ![Net of a cuboid](),Identify the net of a prism,Nets,Which is the correct net for this shape? ![Net of a cuboid](),A,2451,85,AnswerCText,![Net of a cube](),MisconceptionCId,1043.0


### Misconception Mapping Example

In [6]:
misconception_mapping.head()

,MisconceptionId,MisconceptionName
0,0,Does not know that angles in a triangle sum to...
1,1,Uses dividing fractions method for multiplying...
2,2,Believes there are 100 degrees in a full turn
3,3,Thinks a quadratic without a non variable term...
4,4,Believes addition of terms and powers of terms...


## Define Metrics
As metrics I use MAP@25 the same way as it is used for submission evaluation at the Kaggle competition.

In [3]:
# Credit: https://www.kaggle.com/code/abdullahmeda/eedi-map-k-metric

def apk(actual, predicted, k=25):
    """
    Computes the average precision at k.
    
    This function computes the average prescision at k between two lists of
    items.
    
    Parameters
    ----------
    actual : list
             A list of elements that are to be predicted (order doesn't matter)
    predicted : list
                A list of predicted elements (order does matter)
    k : int, optional
        The maximum number of predicted elements
        
    Returns
    -------
    score : double
            The average precision at k over the input lists
    """
    if not actual:
        return 0.0
    if len(predicted)>k:
        predicted = predicted[:k]

    score = 0.0
    num_hits = 0.0

    for i,p in enumerate(predicted):
        # first condition checks whether it is valid prediction
        # second condition checks if prediction is not repeated
        if p in actual and p not in predicted[:i]:
            num_hits += 1.0
            score += num_hits / (i+1.0)
          

    return score / min(len(actual), k)

def mapk(actual, predicted, k=25):
    """
    Computes the mean average precision at k.
    
    This function computes the mean average prescision at k between two lists
    of lists of items.
    
    Parameters
    ----------
    actual : list
             A list of lists of elements that are to be predicted 
             (order doesn't matter in the lists)
    predicted : list
                A list of lists of predicted elements
                (order matters in the lists)
    k : int, optional
        The maximum number of predicted elements
        
    Returns
    -------
    score : double
            The mean average precision at k over the input lists
    """
    
    return np.mean([apk(a,p,k) for a,p in zip(actual, predicted)])

## Load Model

In [5]:
# Load the model
device = "cuda:0" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("/kaggle/input/bge-large-sentence-fine-tune/pytorch/default/1")
model.to(device)

SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': True}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 1024, 'pooling_mode_cls_token': True, 'pooling_mode_mean_tokens': False, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

## Create values to embedd
Values of all columns containing string values are concatenated in the test dataset so they can be embedded.

So values of columns called `WholeAssignment` and `Answer_value` are concatenated into a column called `all_text`. This new column will now contain the assignment and also the answer of a given math question.

In [7]:
df['all_text'] = df['WholeAssignment'] + '. ' + df['Answer_value']
print(f'Example 1 of all_text value: \n{df["all_text"].iloc[0]}\n')
print(f'Example 2 of all_text value: \n{df["all_text"].iloc[1]}\n')
print(f'Example 3 of all_text value: \n{df["all_text"].iloc[2]}\n')

Example 1 of all_text value: 
Express a non-linear equation as a function machine. Function Machines. Which function machine matches the equation \( y=x^{2}+4 ? \). ![A function machine which has 4 parts joined by arrows pointing from left to right. "y" is the first part, written on the left, followed by a horizontal arrow pointing to a rectangle that has "+ 4" written inside it, followed by a horizontal arrow pointing to a rectangle that has "square" written inside it, followed by a horizontal arrow pointing to "𝑥"]()

Example 2 of all_text value: 
Identify a unit of length. Volume and Capacity Units. George has answered a question on perimeter and worked out an answer of \( 310 \).

Behind the star he has written the units that he used.

Which of the following units is definitely wrong? \( 310 \color{yellow} \bigstar\). \( f t \)

Example 3 of all_text value: 
Identify the net of a prism. Nets. Which is the correct net for this shape? ![Net of a cuboid](). ![Net of a cube]()



## Concat `QuestionId` and `Answer` column values
Concat `QuestionId` and `Answer` column values into a single column called `QuestionId_Answer` as it is requested for submission.

In [7]:
df['QuestionId_Answer'] = df['QuestionId'].astype(str) + '_' + df['Answer'].str.replace('Answer', '').str.replace('Text', '')

So the test dataframe will now look the following:

In [8]:
display(HTML(f'<div class="dataframe-container">{df.head(3).to_html(index=False)}</div>'))

QuestionId,WholeAssignment,CorrectAnswer,ConstructId,SubjectId,Answer,Answer_value,Misconception,MisconceptionId,QuestionId_Answer
1758,Express a non-linear equation as a function machine. Function Machines. Which function machine matches the equation \( y=x^{2}+4 ? \),C,1508,649,AnswerDText,"![A function machine which has 4 parts joined by arrows pointing from left to right. ""y"" is the first part, written on the left, followed by a horizontal arrow pointing to a rectangle that has ""+ 4"" written inside it, followed by a horizontal arrow pointing to a rectangle that has ""square"" written inside it, followed by a horizontal arrow pointing to ""𝑥""]()",MisconceptionDId,2389.0,1758_D
1426,Identify a unit of length. Volume and Capacity Units. George has answered a question on perimeter and worked out an answer of \( 310 \).\n\nBehind the star he has written the units that he used.\n\nWhich of the following units is definitely wrong? \( 310 \color{yellow} \bigstar\),D,2012,197,AnswerCText,\( f t \),MisconceptionCId,719.0,1426_C
1181,Identify the net of a prism. Nets. Which is the correct net for this shape? ![Net of a cuboid](),A,2451,85,AnswerCText,![Net of a cube](),MisconceptionCId,1043.0,1181_C


Where columns:

- `all_text`: Strings from columns `WholeAssignment` and `Answer_value` concatenated
- `MisconceptionId`: Ground truth misconception for a given `QuestionId_Answer`


## Create embedding and compute similarities
Embeddings for `all_text` values from test dataset and for `MisconceptionName` values from misconception mapping dataset are created.

Then I find the top 25 most similar `MisconceptionName` embeddings for a single `all_text` embedding using either semantic search or cosine similarities and compare the results.

These strings are embedded using bge-large-en-1.5 model which has fine-tuned beforehand on train dataset. For more detail how the fine tuning was done check out (`fine_tuning_bge_large.html` file).

In [ ]:
top25ids_semantic = []
top25ids_cosine = []

embedding_query = model.encode(df['all_text'], convert_to_tensor=True, normalize_embeddings=True)
embedding_Misconception = model.encode(misconception_mapping.MisconceptionName.values, convert_to_tensor=True, normalize_embeddings=True)
top25ids_semantic = util.semantic_search(embedding_query, embedding_Misconception, top_k=25)
    
embedding_query = model.encode(df['all_text'], convert_to_tensor=False, normalize_embeddings=True)
embedding_Misconception = model.encode(misconception_mapping.MisconceptionName.values, convert_to_tensor=False, normalize_embeddings=True)
    
similarities = cosine_similarity(embedding_query, embedding_Misconception)
# Get top 25 indices for each query
for sim_row in similarities:
    top_indices = np.argsort(sim_row)[-25:][::-1]  # Sort in descending order
    top_matches = [{'corpus_id': int(idx), 'score': float(sim_row[idx])} for idx in top_indices]
    top25ids_cosine.append(top_matches)

Both embeddings have a dimension of 1024. The exact embedding shapes are the following:

In [1]:
print(f"Shape of all_text embedding: [656, 1024]")
print(f"Shape of misconception mapping embedding: [2587, 1024]")

Shape of all_text embedding: [656, 1024]
Shape of misconception mapping embedding: [2587, 1024]


## Create Submission Dataset and Compare Results
Create dataset for submission and compare results of using either sematic search or cosine similarity using metrics as in competition.

### Submission Dataset
Submission dataset contains three columns:
- `QuestionId_Answer`: Combination of QuestionId and answer on this question.
- `MisconceptionId_Semantic`: Ids of 25 most similar misconceptions for given `QuestionId_Answer` based on semantic search
- `MisconceptionId_Cosine`: Ids of 25 most similar misconceptions for given `QuestionId_Answer` based on cosine similarity

In [11]:
df_submission = {}
df_submission['QuestionId_Answer'] = df['QuestionId_Answer'].values
df_submission["MisconceptionId_Semantic"] = [" ".join([str(x["corpus_id"]) for x in top25id]) for top25id in top25ids_semantic]
df_submission["MisconceptionId_Cosine"] = [" ".join([str(x["corpus_id"]) for x in top25id]) for top25id in top25ids_cosine]


df_submission = pd.DataFrame(df_submission)

In [13]:
df_submission.head()

,QuestionId_Answer,MisconceptionId_Semantic,MisconceptionId_Cosine
0,1758_D,168 208 487 593 2496 2511 400 438 577 2291 109...,168 208 487 593 2496 2511 400 438 577 2291 109...
1,1426_C,441 996 2529 1520 1725 1210 719 1244 1183 877 ...,441 996 2529 1520 1725 1210 719 1244 1183 877 ...
2,1181_C,690 1903 2025 1927 2521 1263 2536 402 1959 206...,690 1903 2025 1927 2521 1263 2536 402 1959 206...
3,1451_D,142 753 104 523 1198 10 1925 1164 2407 889 111...,142 753 104 523 1198 10 1925 1164 2407 889 111...
4,819_B,2167 1282 1435 2019 1855 1773 303 2157 1274 24...,2167 1282 1435 2019 1855 1773 303 2157 1274 24...


### Results
Results of using semantic search and of using cosine similarity:

In [12]:
if not IS_SUBMISSION:
    predicted_semantic = df_submission["MisconceptionId_Semantic"].apply(lambda x: [int(y) for y in x.split()])
    predicted_cosine = df_submission["MisconceptionId_Cosine"].apply(lambda x: [int(y) for y in x.split()])
    label = df["MisconceptionId"].apply(lambda x: [x])
    print("Validation of semantic similarity: ", mapk(label, predicted_semantic))
    print("Validation of cosine similarity: ", mapk(label, predicted_cosine))

Validation of semantic similarity:  0.20884444380185
Validation of cosine similarity:  0.20884444380185


As seen from results above, there is no difference in using either semantic search or cosine similarity on the performance.

Also score of approximately 0.208, does not show amazing result. Therefore I assume more has to be done, then just simple similarity search.

In the next notebook, I try to use pre-trained LLM to generate my own misconceptions which I will then try to map onto existing misconceptions using similarity search.